# WiFi Signal Strength Model Training

This notebook trains a Random Forest Regressor using the dataset:

`/home/aidev/project/wifi_signal_prediction_project/data/simulated_wifi_dataset.csv`

It saves the trained model to:

`/home/aidev/project/wifi_signal_prediction_project/models/wifi_signal_model.joblib`

## 1. Import Required Libraries

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import train_test_split


## 2. Set Project Paths

In [ ]:
PROJECT_DIR = Path('/home/aidev/project/wifi_signal_prediction_project')
DATA_DIR = PROJECT_DIR / 'data'
MODEL_DIR = PROJECT_DIR / 'models'

DATA_PATH = DATA_DIR / 'simulated_wifi_dataset.csv'
MODEL_PATH = MODEL_DIR / 'wifi_signal_model.joblib'

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Project Directory:', PROJECT_DIR)
print('Dataset Path:', DATA_PATH)
print('Model Path:', MODEL_PATH)


## 3. Load Dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found at: {DATA_PATH}')

df = pd.read_csv(DATA_PATH)

print('Dataset loaded successfully.')
print('Rows and columns:', df.shape)
df.head()


## 4. Check Dataset Columns

In [ ]:
print('Columns in dataset:')
print(df.columns.tolist())

print('\nDataset information:')
df.info()


## 5. Fix Column Names If Needed

This cell supports both versions of your dataset:

1. `frequency_ghz`, `congestion_percent`, `signal_dbm`
2. `frequency_band`, `interference_level`, `connected_devices`, `signal_strength_dbm`


In [ ]:
df.columns = df.columns.str.strip()

# If old dataset column names exist, rename them to the names used by this model
rename_map = {
    'frequency_band': 'frequency_ghz',
    'interference_level': 'congestion_percent',
    'signal_strength_dbm': 'signal_dbm',
}

df = df.rename(columns=rename_map)

# If obstacles column does not exist, create it using walls as a simple estimate
if 'obstacles' not in df.columns:
    df['obstacles'] = df['walls']

# If congestion_percent came from interference_level 0-10, convert it to 0-100
if 'congestion_percent' in df.columns and df['congestion_percent'].max() <= 10:
    df['congestion_percent'] = df['congestion_percent'] * 10

print('Updated columns:')
print(df.columns.tolist())
df.head()


## 6. Define Features and Target

In [ ]:
FEATURE_COLUMNS = [
    'distance_m',
    'walls',
    'obstacles',
    'frequency_ghz',
    'bandwidth_mhz',
    'congestion_percent',
]

TARGET_COLUMN = 'signal_dbm'

missing_columns = [col for col in FEATURE_COLUMNS + [TARGET_COLUMN] if col not in df.columns]

if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

print('All required columns are available.')


## 7. Clean Dataset

In [ ]:
# Keep only required columns
df = df[FEATURE_COLUMNS + [TARGET_COLUMN]].copy()

# Convert all columns to numeric
for col in FEATURE_COLUMNS + [TARGET_COLUMN]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Missing values before cleaning:')
print(df.isna().sum())

# Fill missing feature values with median
for col in FEATURE_COLUMNS:
    df[col] = df[col].fillna(df[col].median())

# Remove rows where target is missing
df = df.dropna(subset=[TARGET_COLUMN])

print('\nMissing values after cleaning:')
print(df.isna().sum())

print('\nFinal dataset shape:', df.shape)
df.head()


## 8. Split Data

In [ ]:
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

print('Training rows:', X_train.shape[0])
print('Testing rows:', X_test.shape[0])


## 9. Train Random Forest Regressor

In [ ]:
model = RandomForestRegressor(
    n_estimators=250,
    random_state=42,
    max_depth=18,
    min_samples_split=3,
    n_jobs=-1,
)

model.fit(X_train, y_train)

print('Training completed successfully.')


## 10. Evaluate Model

In [ ]:
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f'Mean Absolute Error: {mae:.2f} dBm')
print(f'RMSE: {rmse:.2f} dBm')
print(f'R2 Score: {r2:.4f}')


## 11. Save Model

In [ ]:
model_bundle = {
    'model': model,
    'feature_columns': FEATURE_COLUMNS,
    'target_column': TARGET_COLUMN,
    'mae': mae,
    'rmse': rmse,
    'r2': r2,
}

joblib.dump(model_bundle, MODEL_PATH)

print('Model saved successfully.')
print('Saved at:', MODEL_PATH)


## 12. Test Prediction

In [ ]:
sample_input = pd.DataFrame([{
    'distance_m': 10,
    'walls': 2,
    'obstacles': 3,
    'frequency_ghz': 2.4,
    'bandwidth_mhz': 40,
    'congestion_percent': 35,
}])

sample_prediction = model.predict(sample_input)[0]

print(f'Predicted WiFi Signal Strength: {sample_prediction:.2f} dBm')
